# Amazon E-Commerce Anomaly Detection — Live Demo

**Primary Architecture:** Amazon TGAT (Temporal Graph Attention) via Contrastive Link Prediction  
**Test F1:** 77.01% (Self-supervised anomaly detection)  
**Historical Baseline:** IEEE-CIS E10 CatBoost Ensemble (62.25% F1)  

This notebook clones the repository, loads the frozen Amazon TGAT checkpoint (and E10 baseline), starts the FastAPI backend, and creates a public Command Center dashboard via ngrok.

In [ ]:
def verify_models():
    print("Step 2: Verifying Amazon TGAT and Baseline E10 model artifacts...")
    required = {
        "models/amazon_tgat.pt":         0.01,
        "models/e10_base.cbm":          100,
        "models/e10_deep.cbm":          100,
        "app/amazon_predictor.py":       0,
    }
    ok = True
    for f, min_mb in required.items():
        if not os.path.isfile(f):
            print(f"\u274c Missing: {f}")
            ok = False
        elif min_mb > 0 and os.path.getsize(f) / 1e6 < min_mb:
            size = os.path.getsize(f) / 1e6
            print(f"\u274c {f} too small ({size:.2f}MB < {min_mb}MB) - LFS pull failed?")
            ok = False
        else:
            size = os.path.getsize(f) / 1e6
            print(f"\u2705 {f} ({size:.2f} MB)")
    return ok

if not verify_models():
    raise RuntimeError("Model verification failed. Check Git LFS pull.")


In [ ]:
print("Step 3: Installing dependencies...")
!pip install -q -r requirements.txt
print("\u2705 Dependencies installed.")


In [ ]:
import threading
import time
import requests
import uvicorn
import json
from app.main import app
from pyngrok import ngrok

print("Step 4: Authenticating ngrok...")
token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    token = os.environ.get('NGROK_AUTHTOKEN')

if not token:
    print("\u274c NGROK_AUTHTOKEN not found.")
    print("  1. Click the key icon in the Colab sidebar.")
    print("  2. Add secret: Name=NGROK_AUTHTOKEN, Value=<your_token>")
    print("  3. Enable 'Notebook access', then re-run this cell.")
    print("  Get your free token at: https://dashboard.ngrok.com")
else:
    print("\u2705 NGROK_AUTHTOKEN found.")

print("\nStep 5: Starting FastAPI (Amazon TGAT)...")
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("Step 6: Waiting for API health check...")
for i in range(20):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        if r.status_code == 200:
            health = r.json()
            amazon_ok = health['primary_system']['loaded']
            print(f"\u2705 FastAPI ONLINE — Amazon TGAT: {'OK' if amazon_ok else 'FAIL'}")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("\u274c FastAPI did not start within 20 seconds.")

if token:
    print("\nStep 7: Creating ngrok tunnel...")
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url

    print("\n" + "="*50)
    print("  AMAZON E-COMMERCE ANOMALY DETECTION DEMO")
    print("="*50)
    print(f"  Dashboard: {public_url}")
    print(f"  API Docs:  {public_url}/docs")
    print("="*50)

    print("\nStep 8: Smoke test (Amazon TGAT Inference)...")
    sample = {
        "reviewerID": "A34BZN6YJG123", 
        "asin": "B0002L5R78", 
        "overall": 5.0, 
        "unixReviewTime": time.time()
    }
    try:
        resp = requests.post("http://127.0.0.1:8000/predict/amazon", json=sample, timeout=15)
        if resp.status_code == 200:
            pred = resp.json()
            print(f"\u2705 Smoke test passed!")
            print(f"   Prediction:         {pred['prediction']}")
            print(f"   Anomaly Prob:       {pred['anomaly_probability']}")
            print(f"   Risk Level:         {pred['risk_level']}")
            print(f"   User Half-Life:     {pred['temporal_half_life_user_days']} days")
            print(f"   Product Half-Life:  {pred['temporal_half_life_product_days']} days")
        else:
            print(f"\u274c Smoke test failed: HTTP {resp.status_code}")
    except Exception as e:
        print(f"\u274c Smoke test exception: {e}")
else:
    print("\n\u26a0 Skipping ngrok (no token). API is running locally on port 8000.")